In [5]:
from langchain_community.document_loaders import ArxivLoader
import arxiv

In [16]:
client = arxiv.Client()

search = arxiv.Search(
    query="self attention",
    max_results=1,
)

In [21]:
paper = list(client.results(search))

In [30]:
paper

arxiv.Result(entry_id='http://arxiv.org/abs/2605.18805v1', updated=datetime.datetime(2026, 5, 11, 18, 55, 32, tzinfo=datetime.timezone.utc), published=datetime.datetime(2026, 5, 11, 18, 55, 32, tzinfo=datetime.timezone.utc), title='RecoAtlas: From Semantic Plausibility to Set-Level Utility in LLM Recommendation Agents', authors=[arxiv.Result.Author('Imad Aouali'), arxiv.Result.Author('Flavian Vasile'), arxiv.Result.Author('Otmane Sakhi'), arxiv.Result.Author('Alexandre Gilotte'), arxiv.Result.Author('Benjamin Heymann')], summary='LLM recommendation agents increasingly produce structured recommendation reports: sets of items accompanied by natural-language justifications. Yet existing evaluations often reduce this setting to reranking small shortlisted candidate sets or judge reports mainly by semantic plausibility. We introduce Recommendation Atlas (Agentic Tool-Level Assessment for Shopping), or RecoAtlas, a benchmark and toolkit for evaluating shopping agents with behavior-grounded m

In [29]:
paper.entry_id

'http://arxiv.org/abs/2605.18805v1'

In [25]:
pdf_path = paper.download_pdf(dirpath="./papers")

AttributeError: 'list' object has no attribute 'download_pdf'

In [26]:
import arxiv

client = arxiv.Client()

search = arxiv.Search(
    query="LLM agents",
    max_results=1,
)

paper = next(client.results(search))

print(type(paper))
print(paper.pdf_url)

<class 'arxiv.Result'>
https://arxiv.org/pdf/2605.18805v1


In [39]:
import os
import arxiv
import requests
import tempfile
from langchain_community.document_loaders import PyMuPDFLoader

search = arxiv.Search(query="attention is all you need", max_results=1)
paper = next(arxiv.Client().results(search))

pdf_url = paper.entry_id.replace("/abs/", "/pdf/") + ".pdf"

tmp = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
tmp.write(requests.get(pdf_url).content)
tmp.close()

try:
    loader = PyMuPDFLoader(tmp.name)
    docs = loader.load()
    print(docs[0].page_content, docs[0].metadata)
finally:
    os.remove(tmp.name)  

Do You Even Need Attention? A Stack of Feed-Forward Layers Does
Surprisingly Well on ImageNet
Luke Melas-Kyriazi
Oxford University
lukemk@robots.ox.ac.uk
Abstract
The strong performance of vision transformers on im-
age classiﬁcation and other vision tasks is often attributed
to the design of their multi-head attention layers. How-
ever, the extent to which attention is responsible for this
strong performance remains unclear. In this short report,
we ask: is the attention layer even necessary? Speciﬁcally,
we replace the attention layer in a vision transformer with
a feed-forward layer applied over the patch dimension. The
resulting architecture is simply a series of feed-forward lay-
ers applied over the patch and feature dimensions in an al-
ternating fashion. In experiments on ImageNet, this archi-
tecture performs surprisingly well: a ViT/DeiT-base-sized
model obtains 74.9% top-1 accuracy, compared to 77.9%
and 79.9% for ViT and DeiT respectively. These results in-
dicate that aspe

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from src.models import btw_retrieval
from dotenv import load_dotenv

from tavily import TavilyClient
load_dotenv()
model = ChatOpenAI(model = 'gpt-4o-mini')

def handler(query):
    
    
    route_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Decide if answering this question requires a real-time web search (recent events, "
         "current prices, breaking news) or if your general knowledge is sufficient."),
        ("human", "{query}"),
    ])
    
    dec = (route_prompt | model.with_structured_output(btw_retrieval)).invoke({'query':query}).need_web_retrieval
    
    if dec:
        client = TavilyClient()
        resp = client.search(
            query = query,
            max_results=3
        ).get('results',[])
        
        context = "\n\n".join(con['content'] for con in resp)
        sources = "\n".join(f"- {r['url']}" for r in resp)
        
        answer_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Answer the question using the web search results below. Be concise.\n\n"
             "Results:\n{context}\n\nSources:\n{sources}"),
            ("human", "{query}"),
        ])
        kwargs = {'query':query,'context':context,'sources':sources}
    else:
        answer_prompt = ChatPromptTemplate.from_messages([
            ("system","Answer the question from your general knowladge but never add any false information. Be concise.\n\n"),
            ("human", "{query}"),
        ])
        kwargs = {'query':query}
        
    return (answer_prompt | model).invoke(kwargs).content

In [10]:
handler('what is the temp of today in dhaka')

'The current temperature in Dhaka is 33° Celsius (or 91° Fahrenheit).'

In [4]:
from datasets import load_dataset

dataset = load_dataset("galileo-ai/ragbench","covidqa", split="test")

len(dataset[0])

26